# CSC 303: Software Engineering & AI Applications
## Academic Advisor Machine Learning Pipeline & Model Serialization
**Lecturer:** Dr. Felix  
**Student:** Paul (Academic Advisor System)  
**Course Code:** CSC 303  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/username/csc303-academic-advisor/blob/main/notebooks/csc303_academic_advisor_model_training.ipynb)

---
### Project Objective
Using the strategies employed in class, this notebook develops an intelligent Academic Advisor AI System combining rigorous software engineering standards with machine learning techniques:
1. **Student Risk Classification**: Identifies academic risk status (`Low Risk`, `Moderate Risk`, `High Risk`) using Random Forest ensembles.
2. **GPA Trajectory Regression**: Projects next-term GPA using Gradient Boosting to guide course load optimization.
3. **Prerequisite & Credit Load Auditing**: Formulates rule-based guidance integrated with ML feature importances.
4. **Artifact Serialization**: Exports full preprocessing + model pipelines into a portable `.pkl` file for web framework deployment.

### Step 1: Environment Setup & Library Imports

In [ ]:
# Install and import dependencies
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, r2_score, mean_squared_error

print("Libraries successfully imported!")

### Step 2: Realistic Academic Advising Dataset Synthesis
We model authentic undergraduate Computer Science dynamics: lecture/lab attendance rates, independent study hours, credit loads, prerequisite failures, and stress indices.

In [ ]:
np.random.seed(42)
n_samples = 2500

levels = ["100L", "200L", "300L", "400L"]
tracks = ["Software Engineering", "Artificial Intelligence & Data Science", "Cybersecurity", "Networking & Systems"]

level_choices = np.random.choice(levels, size=n_samples, p=[0.25, 0.28, 0.27, 0.20])
track_choices = np.random.choice(tracks, size=n_samples, p=[0.40, 0.30, 0.18, 0.12])

completed_credits = []
for lvl in level_choices:
    if lvl == "100L": completed_credits.append(np.random.randint(0, 24))
    elif lvl == "200L": completed_credits.append(np.random.randint(24, 64))
    elif lvl == "300L": completed_credits.append(np.random.randint(64, 104))
    else: completed_credits.append(np.random.randint(104, 140))

study_hours = np.clip(np.random.normal(loc=16, scale=7, size=n_samples), 2, 45)
attendance_rate = np.clip(np.random.normal(loc=82, scale=14, size=n_samples), 35, 100)
current_credit_load = np.random.choice([12, 15, 18, 21, 24], size=n_samples, p=[0.10, 0.25, 0.40, 0.18, 0.07])
extracurricular_hours = np.clip(np.random.exponential(scale=6, size=n_samples), 0, 30)
stress_level = np.clip(
    np.round(1 + (current_credit_load / 6) + (extracurricular_hours / 10) - (study_hours / 15) + np.random.normal(0, 0.8, n_samples)),
    1, 5
).astype(int)

base_ability = (study_hours / 35.0) * 1.5 + (attendance_rate / 100.0) * 1.8 - (stress_level / 5.0) * 0.5 + np.random.normal(0, 0.3, n_samples)
current_gpa = np.clip(np.round(base_ability + 0.8, 2), 1.20, 4.00)
core_cs_avg_grade = np.clip(np.round(current_gpa + np.random.normal(-0.1, 0.25, n_samples), 2), 1.00, 4.00)

failed_prob = np.clip((4.0 - core_cs_avg_grade) * 0.25 + (100 - attendance_rate) * 0.015, 0, 0.9)
failed_courses = np.random.binomial(n=4, p=failed_prob / 4)

risk_labels = []
projected_gpas = []

for i in range(n_samples):
    g = current_gpa[i]
    att = attendance_rate[i]
    sh = study_hours[i]
    fc = failed_courses[i]
    load = current_credit_load[i]
    stress = stress_level[i]

    delta = (sh - 15) * 0.03 + (att - 80) * 0.015 - (load - 18) * 0.04 - (fc * 0.15) - (stress - 3) * 0.05
    proj_gpa = np.clip(round(g + delta + np.random.normal(0, 0.12), 2), 1.00, 4.00)
    projected_gpas.append(proj_gpa)

    if g < 2.30 or att < 65.0 or fc >= 2 or (load > 21 and g < 2.70):
        risk_labels.append("High Risk")
    elif g >= 3.30 and att >= 80.0 and fc == 0 and load <= 21:
        risk_labels.append("Low Risk")
    else:
        risk_labels.append("Moderate Risk")

df = pd.DataFrame({
    "academic_level": level_choices,
    "specialization_track": track_choices,
    "completed_credits": completed_credits,
    "current_credit_load": current_credit_load,
    "current_gpa": current_gpa,
    "core_cs_avg_grade": core_cs_avg_grade,
    "attendance_rate": np.round(attendance_rate, 1),
    "study_hours_per_week": np.round(study_hours, 1),
    "failed_courses_count": failed_courses,
    "extracurricular_hours": np.round(extracurricular_hours, 1),
    "stress_level": stress_level,
    "risk_status": risk_labels,
    "projected_next_gpa": projected_gpas
})

print(f"Dataset synthesized: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

### Step 3: Exploratory Data Analysis (EDA)
Inspect the target distributions and key pedagogical relationships.

In [ ]:
print("Risk Status Distribution:")
print(df["risk_status"].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x="risk_status", palette="viridis", ax=axes[0])
axes[0].set_title("Distribution of Academic Risk Tiers")

sns.scatterplot(data=df, x="study_hours_per_week", y="current_gpa", hue="risk_status", palette="viridis", ax=axes[1])
axes[1].set_title("Study Hours vs. CGPA by Risk Level")
plt.tight_layout()
plt.show()

### Step 4: Pipeline Construction with ColumnTransformer
We standardize numeric features and one-hot encode categorical degree tracks.

In [ ]:
categorical_features = ["academic_level", "specialization_track"]
numeric_features = [
    "completed_credits",
    "current_credit_load",
    "current_gpa",
    "core_cs_avg_grade",
    "attendance_rate",
    "study_hours_per_week",
    "failed_courses_count",
    "extracurricular_hours",
    "stress_level"
]

X = df[categorical_features + numeric_features]
y_class = df["risk_status"]
y_reg = df["projected_next_gpa"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)
_, _, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.20, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples, Testing set: {X_test.shape[0]} samples")

### Step 5: Model Experimentation & Cross-Validation
Compare Logistic Regression, Decision Tree, and Random Forest Classifier across 5-fold cross validation.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=160, max_depth=10, random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("clf", model)])
    scores = cross_val_score(pipe, X_train, y_train_cls, cv=5, scoring="accuracy")
    print(f"{name:20s} Mean CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")

### Step 6: Final Pipelines Training & Evaluation

In [ ]:
# Final Classifier Pipeline
classifier_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=160, max_depth=10, min_samples_split=4, random_state=42, class_weight="balanced"
    ))
])

# Final Regressor Pipeline for GPA Trajectory
regressor_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42))
])

classifier_pipeline.fit(X_train, y_train_cls)
regressor_pipeline.fit(X_train, y_train_reg)

y_pred_cls = classifier_pipeline.predict(X_test)
print("=== CLASSIFICATION REPORT (RISK STATUS) ===")
print(classification_report(y_test_cls, y_pred_cls))

y_pred_reg = regressor_pipeline.predict(X_test)
print(f"GPA Regression R^2 Score: {r2_score(y_test_reg, y_pred_reg):.4f}")
print(f"GPA Regression RMSE:     {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}")

### Step 7: Feature Importances & Pedagogical Insights

In [ ]:
rf_model = classifier_pipeline.named_steps["classifier"]
cat_encoder = classifier_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_features = list(cat_encoder.get_feature_names_out(categorical_features))
all_features = numeric_features + encoded_cat_features

feat_importances = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_importances.head(8).plot(kind="barh", color="#6366f1")
plt.title("Top 8 Most Predictive Features in Academic Advising System")
plt.xlabel("Relative Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Step 8: Serialization to .PKL File
We package the complete trained pipelines and metadata into `academic_advisor_model.pkl` for deployment with our web application.

In [ ]:
os.makedirs("../models", exist_ok=True)
pkl_output_path = "../models/academic_advisor_model.pkl"

artifact_payload = {
    "metadata": {
        "course": "CSC 303: Software Engineering & AI Applications",
        "assignment": "Dr. Felix Academic Advisor System",
        "model_version": "1.0.0",
        "model_type": "Composite Ensemble (RandomForest + GradientBoosting)",
        "classes": list(classifier_pipeline.classes_),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features
    },
    "classifier_pipeline": classifier_pipeline,
    "regressor_pipeline": regressor_pipeline,
    "metrics": {
        "classification_accuracy": float(accuracy_score(y_test_cls, y_pred_cls)),
        "regression_r2": float(r2_score(y_test_reg, y_pred_reg)),
        "feature_importances": feat_importances.to_dict()
    }
}

with open(pkl_output_path, "wb") as f:
    pickle.dump(artifact_payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Successfully exported .PKL model artifact to: {pkl_output_path}")
print(f"File size: {os.path.getsize(pkl_output_path) / (1024*1024):.2f} MB")

### Step 9: Google Colab Direct Download Cell (Optional for Colab Users)
If running on Google Colab, executing the cell below automatically triggers a browser download of the generated `.pkl` file.

In [ ]:
try:
    from google.colab import files
    files.download(pkl_output_path)
    print("Triggered Colab browser download for .PKL artifact!")
except ImportError:
    print("Running outside Google Colab (Local Environment) - download step bypassed.")

### Step 10: Validation of .PKL Loading & Downstream Inference
Simulate loading the `.pkl` artifact from disk to confirm deployment compatibility.

In [ ]:
with open(pkl_output_path, "rb") as f:
    loaded_artifact = pickle.load(f)

clf_pipeline = loaded_artifact["classifier_pipeline"]
reg_pipeline = loaded_artifact["regressor_pipeline"]

sample_student = pd.DataFrame([{
    "academic_level": "300L",
    "specialization_track": "Software Engineering",
    "completed_credits": 78,
    "current_credit_load": 18,
    "current_gpa": 3.52,
    "core_cs_avg_grade": 3.60,
    "attendance_rate": 88.5,
    "study_hours_per_week": 20.0,
    "failed_courses_count": 0,
    "extracurricular_hours": 8.0,
    "stress_level": 2
}])

pred_risk = clf_pipeline.predict(sample_student)[0]
pred_gpa = reg_pipeline.predict(sample_student)[0]

print("=== INFERENCE VERIFICATION ===")
print(f"Sample Student Profile: Paul (300L CS, Software Engineering Track)")
print(f"Predicted Risk Standing: {pred_risk}")
print(f"Projected Next GPA:      {pred_gpa:.2f}")
print("All systems verified for web deployment!")